# 08. Controlled ablations and attack views

This notebook rebuilds, from the study's saved ablation runs:

- **Table VI** and **Tables S-XXXIV to S-XXXVI** (`reports/causal/`), **Figures S13 and S14**: why trees are easier
  to evade, from controlled architecture ablations;
- **Figure 2**, **Figures S15 to S18** and **Table S-XXXVII** (`reports/attack_views/`), with the numbers the text
  quotes about them (`reports/attack_views/attack_view_numbers.csv`): what an evasion looks like.

Inputs: `runs/<dataset>/causal/out_<variant>_seed<s>_<mode>.npz` and `clean_<variant>_seed<s>.json`
(fraudadv.causal.ablate) and `runs/<dataset>/causal/views.pkl` (fraudadv.report.attack_views compute) on Sparkov,
TabFormer, IEEE-CIS and BAF; `data/processed/<dataset>/meta.json`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))   # the library (not needed after pip install -e .)
from fraudadv import notebook as nb
ROOT = nb.setup()          # works from the folder root, sets the study's environment

The output folder, created up front as `scripts/make_results.sh` does.

In [ ]:
(ROOT / "reports" / "causal").mkdir(parents=True, exist_ok=True)

## Table VI, Tables S-XXXIV to S-XXXVI, Figures S13 and S14: controlled ablations

`fraudadv.causal.analyze.main` collects the per-fraud outcomes of every ablation variant
(`reports/causal/per_fraud_outcomes.csv.gz`), estimates each intervention's effect with paired bootstrap intervals
(`contrasts.csv`), fits the factorial linear probability model (`factorial_lpm.csv`) and gathers the clean metrics
(`clean_metrics.csv`); it then calls `fraudadv.causal.report.write_all`, which writes the clean performance of each
variant (`variants_clean.csv`) and the effects derived from the factorial model (`factorial_effects.csv`) and draws
the two figures (and `dose_response.csv`). It prints the pooled constrained effects and the factorial model.

In [ ]:
from fraudadv.causal import analyze

analyze.main()

In [ ]:
import pandas as pd

C = pd.read_csv(ROOT / "reports/causal/contrasts.csv")
C[C["dataset"] == "pooled"].head(12)          # Table VI and the pooled rows of Table S-XXXV

Table S-XXXIV: clean performance of every ablation variant, mean over datasets with each variant.

In [ ]:
nb.show_csv("reports/causal/variants_clean.csv", rows=14)

Table S-XXXV: every intervention's effect, per dataset and pooled (all rows of `contrasts.csv`).

In [ ]:
nb.show_csv("reports/causal/contrasts.csv", rows=12)

Table S-XXXVI: the factorial model (representation x learned function), and the effects derived from it.

In [ ]:
nb.show_csv("reports/causal/factorial_lpm.csv", rows=8)
nb.show_csv("reports/causal/factorial_effects.csv")

**Figure S13** (`fig_causal_effects`): pooled effect of each intervention on the ASR per attack, and on clean recall, with 95% bootstrap intervals.

In [ ]:
nb.show_figure("fig_causal_effects")

**Figure S14** (`fig_causal_dose`): ASR of detected frauds against XGBoost depth and MLP layers, per dataset, constrained, eps 0.25.

In [ ]:
nb.show_figure("fig_causal_dose")

## Figure 2, Figures S15 to S18, Table S-XXXVII: attack views

`fraudadv.report.attack_figures.main` (what `python -m fraudadv.report.attack_views figures` runs) reads the stored
views of the tuned XGBoost and MLP bases and writes Figure 2 and Figures S15 to S18, the k-NN summary, Table S-XXXVII
(`attack_examples.csv`, with the score percentiles before and after in `attack_examples_scores.csv`) and the numbers
the text quotes (`attack_view_numbers.csv`), all in `reports/attack_views/`.

In [ ]:
from fraudadv.report import attack_figures

attack_figures.main()
nb.show_csv("reports/attack_views/attack_view_numbers.csv", rows=12)

**Figure 2** (`fig_attack_views`): what an evasion looks like: slices of one Sparkov fraud for XGBoost and the MLP, distance to the threshold, k-NN fraud share.

In [ ]:
nb.show_figure("fig_attack_views")

**Figure S15** (`fig_attack_slices`): decision-function slices of XGBoost and the MLP.

In [ ]:
nb.show_figure("fig_attack_slices")

**Figure S16** (`fig_attack_distance`): distributions of the distance to the threshold per dataset, setting, model and group.

In [ ]:
nb.show_figure("fig_attack_distance")

**Figure S17** (`fig_attack_map`): UMAP of legit transactions, frauds and both models' white-box adversarial examples.

In [ ]:
nb.show_figure("fig_attack_map")

**Figure S18** (`fig_attack_profile`): mean absolute change per feature for every model and attack.

In [ ]:
nb.show_figure("fig_attack_profile")

Table S-XXXVII: evasion examples in original units.

In [ ]:
nb.show_csv("reports/attack_views/attack_examples.csv", rows=8)
nb.show_csv("reports/attack_views/attack_examples_scores.csv")

## Run it yourself: the decision-function slice of Figure 2

The slice of Figure 2 (Sparkov, constrained) is deterministic: one fraud that both bases detect and both white-box
attacks evade, and each model's logit on a 121 x 121 grid of two attacker-controllable features. This cell recomputes
it from the saved XGBoost and MLP (seed 0) with `fraudadv.report.attack_views.slice_view`, in memory only, and
compares it with the stored views (`runs/sparkov/causal/views.pkl`). The rest of the views uses unseeded
HopSkipJump and UMAP, so it is not recomputed here.

In [ ]:
import pickle

import numpy as np

from fraudadv.attacks.constraints import Constraints
from fraudadv.attacks.run_attacks import RUNS, load_model
from fraudadv.data.load import load
from fraudadv.report.attack_views import BASES, slice_view

a, meta = load("sparkov")
R = pickle.loads((RUNS / "sparkov" / "causal" / "views.pkl").read_bytes())
models = {v: load_model("sparkov", f, 0) for v, f in BASES.items()}
live = slice_view(R, models, Constraints(meta, a["X_train"], "constrained"), list(meta["features"]))
stored = R["slice"]
feats = list(meta["features"])
print(f"fraud {live['i0']} (stored {stored['i0']}), features {feats[live['j']]} x {feats[live['k']]} "
      f"(stored {feats[stored['j']]} x {feats[stored['k']]})")
for v in BASES:
    print(f"{v}: largest |live - stored| logit on the grid {np.abs(live[v]['L'] - stored[v]['L']).max():.2e}; "
          f"smallest in-slice evasion {live[v]['best'][1]:.3f} (stored {stored[v]['best'][1]:.3f})")

## Rerun the runs behind these results from scratch

The calls `scripts/reproduce.sh` makes for the ablations and the attack views (stage `extras`) on one dataset. Both
run on `sparkov`, `tabformer`, `ieee_cis` and `baf` only (not `ulb`). They take hours and write into
`runs/<dataset>/causal/`. Not run here (`RERUN = False`).

In [ ]:
RERUN = False
DATASET = "sparkov"

if RERUN:
    from fraudadv.causal import ablate
    from fraudadv.report import attack_views

    CAUSAL_VARIANTS = ["xgb", "xgb_d2", "xgb_d6", "xgb_d10", "xgb_mono", "xgb_from_mlp", "mlp", "mlp_l1", "mlp_l4",
                       "mlp_relu", "mlp_sn", "mlp_wd", "mlp_from_xgb"]
    ablate.run(DATASET, CAUSAL_VARIANTS, [0, 1, 2], ["constrained", "unconstrained"])   # seeds 0-2, both settings
    attack_views.compute(DATASET)                                                        # runs/<dataset>/causal/views.pkl

## Check: every file this notebook wrote, against the reference build

In [ ]:
figures = ["fig_causal_effects", "fig_causal_dose", "fig_attack_views", "fig_attack_slices", "fig_attack_distance",
           "fig_attack_map", "fig_attack_profile"]
written = [f"reports/causal/{f}" for f in ("per_fraud_outcomes.csv.gz", "contrasts.csv", "factorial_lpm.csv",
                                           "clean_metrics.csv", "dose_response.csv", "variants_clean.csv",
                                           "factorial_effects.csv")]
written += [f"reports/attack_views/{f}.csv" for f in ("attack_examples", "attack_examples_scores", "attack_view_numbers",
                                                     "distance_to_boundary", "knn_fraud_share", "perturbation_profile")]
written += [f"reports/{name}.{ext}" for name in figures for ext in ("pdf", "png")]
nb.check(written)